In [1]:
import os
os.chdir("../")

In [2]:
%pwd

'c:\\Users\\MATT\\Documents\\End-to-end-chicken-Disease-Classification'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    image_data_dir: Path
    label_csv_path: Path
    params_image_size: list
    params_epochs: int
    params_batch_size: int
    params_learning_rate: float
    params_is_augmentation: bool

@dataclass(frozen=True)
class PrepareCallbacksConfig:
    root_dir: Path
    checkpoint_model_filepath: Path
    tensorboard_root_log_dir: Path
    checkpoint_model_dir: Path
    checkpoint_model_filename: Path

In [4]:
import sys
import os

# Add src directory to Python path
src_path = os.path.join(os.getcwd(), 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)


from Chicken_Disease_Classification.utils.common import read_yaml, create_directories 
from Chicken_Disease_Classification.constant import *


In [5]:
import tensorflow as tf
tf.config.run_functions_eagerly(True)
tf.compat.v1.enable_eager_execution()
import time 
import os
from pathlib import Path

[2025-09-30 10:59:57,198: WARNING: module_wrapper: From C:\Users\MATT\AppData\Local\Temp\ipykernel_14932\3091630949.py:3: The name tf.enable_eager_execution is deprecated. Please use tf.compat.v1.enable_eager_execution instead.
]


In [6]:
from pathlib import Path

class TrainingConfig:
    def __init__(self, root_dir: Path, trained_model_path: Path, updated_base_model_path: Path,
                 image_data_dir: Path, label_csv_path: Path,
                 params_image_size: list, params_epochs: int, params_batch_size: int,
                 params_is_augmentation: bool, params_learning_rate: float):
        self.root_dir = root_dir
        self.trained_model_path = trained_model_path
        self.updated_base_model_path = updated_base_model_path
        self.image_data_dir = image_data_dir
        self.label_csv_path = label_csv_path
        self.params_image_size = params_image_size
        self.params_epochs = params_epochs
        self.params_batch_size = params_batch_size
        self.params_is_augmentation = params_is_augmentation
        self.params_learning_rate = params_learning_rate


In [7]:
import os
from pathlib import Path

class ConfigurationManager:
    def __init__(self, 
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([Path(self.config.artifacts_root)])

    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params

        create_directories([Path(training.root_dir)])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            image_data_dir=Path(training.image_data_dir),
            label_csv_path=Path(training.label_csv_path),
            params_image_size=params.IMAGE_SIZE,
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_learning_rate=params.LEARNING_RATE
        )
        
        return training_config

    def get_prepare_callbacks_config(self) -> PrepareCallbacksConfig:
        prepare_callbacks_config = self.config.prepare_callbacks
        model_ckpt_dir = os.path.dirname(prepare_callbacks_config.checkpoint_model_filepath)
        create_directories([
            Path(str(prepare_callbacks_config.root_dir)),
            Path(str(model_ckpt_dir)), 
            Path(str(prepare_callbacks_config.tensorboard_root_log_dir))
        ])

        prepare_callbacks_config = PrepareCallbacksConfig(
            root_dir=Path(prepare_callbacks_config.root_dir),
            checkpoint_model_filepath=Path(prepare_callbacks_config.checkpoint_model_filepath),
            tensorboard_root_log_dir=Path(prepare_callbacks_config.tensorboard_root_log_dir),
            checkpoint_model_dir=Path(prepare_callbacks_config.checkpoint_model_dir),
            checkpoint_model_filename=Path(prepare_callbacks_config.checkpoint_model_filename)
        )

        return prepare_callbacks_config

In [8]:
import os
import time
import tensorflow as tf
from pathlib import Path
from Chicken_Disease_Classification.entity.config_entity import PrepareCallbacksConfig


class PrepareCallback:
    def __init__(self, config: PrepareCallbacksConfig):
        self.config = config

    @property
    def create_tb_callbacks(self):
        """Create TensorBoard callback with timestamped log directory"""
        timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
        tb_running_log_dir = os.path.join(
            str(self.config.tensorboard_root_log_dir),
            f"tb_logs_at_{timestamp}"
        )
        
        # Create directory if it doesn't exist
        os.makedirs(tb_running_log_dir, exist_ok=True)
        
        return tf.keras.callbacks.TensorBoard(log_dir=tb_running_log_dir)

    @property
    def create_ckpt_callbacks(self):
        """Create ModelCheckpoint callback"""
        # Create directory for checkpoint if it doesn't exist
        checkpoint_dir = os.path.dirname(str(self.config.checkpoint_model_filepath))
        os.makedirs(checkpoint_dir, exist_ok=True)
        
        return tf.keras.callbacks.ModelCheckpoint(
            filepath=str(self.config.checkpoint_model_filepath),
            save_best_only=True,
            save_weights_only=False,
            monitor='val_loss',
            mode='min',
            verbose=1
        )

    def get_tb_ckpt_callbacks(self):
        """Get both TensorBoard and ModelCheckpoint callbacks"""
        return [
            self.create_tb_callbacks,
            self.create_ckpt_callbacks
        ]


In [9]:
import os
import time
import pandas as pd
import numpy as np
import tensorflow as tf
from pathlib import Path
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    TensorBoard
)
from tensorflow.keras import regularizers


class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config
        self.model = None
        self.train_generator = None
        self.valid_generator = None

    def get_base_model(self):
        """
        Load the updated base model and recompile with custom settings.
        Add BatchNorm + Dropout + L2 regularization to prevent overfitting.
        """
        # Load pretrained / updated base model
        self.model = tf.keras.models.load_model(self.config.updated_base_model_path)

        # ---- Step 1: Infer number of classes ----
        df = pd.read_csv(self.config.label_csv_path)
        num_classes = df['label'].nunique()

        # ---- Step 2: Adjust last layers if mismatch ----
        if hasattr(self.model.layers[-1], "units") and self.model.layers[-1].units != num_classes:
            print(f"⚡ Adjusting last Dense layer: {self.model.layers[-1].units} ➝ {num_classes}")
            x = self.model.layers[-2].output

            # 🔹 Add Batch Normalization
            x = tf.keras.layers.BatchNormalization()(x)

            # 🔹 Add Dropout
            x = tf.keras.layers.Dropout(0.5)(x)

            # 🔹 Final Dense with L2
            output = tf.keras.layers.Dense(
                num_classes,
                activation="softmax",
                kernel_regularizer=regularizers.l2(0.01)
            )(x)

            self.model = tf.keras.Model(inputs=self.model.input, outputs=output)

        # ---- Step 3: Re-compile ----
        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=self.config.params_learning_rate),
            loss="categorical_crossentropy",
            metrics=["accuracy"]
        )
        print("✅ Model compiled with BatchNorm + Dropout + L2 regularization.")

    def train_valid_generator(self):
        """
        Prepare training and validation generators using the image folder + CSV.
        Includes strong augmentation to reduce overfitting.
        """
        df = pd.read_csv(self.config.label_csv_path)

        # Print dataset distribution
        print("\n📊 Class Distribution in Dataset:")
        print(df['label'].value_counts())

        datagenerator_kwargs = dict(
            rescale=1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = ImageDataGenerator(**datagenerator_kwargs)

        self.valid_generator = valid_datagenerator.flow_from_dataframe(
            dataframe=df,
            directory=self.config.image_data_dir,
            x_col='images',
            y_col='label',
            subset="validation",
            shuffle=False,
            class_mode='categorical',
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = ImageDataGenerator(
                rotation_range=40,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                horizontal_flip=True,
                brightness_range=[0.8, 1.2],  # 🔹 brightness augmentation
                channel_shift_range=20.0,      # 🔹 color augmentation
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_dataframe(
            dataframe=df,
            directory=self.config.image_data_dir,
            x_col='images',
            y_col='label',
            subset="training",
            shuffle=True,
            class_mode='categorical',
            **dataflow_kwargs
        )

        # Print loaded dataset summary
        print(f"\n🖼️ Total training images: {self.train_generator.samples}")
        print(f"🖼️ Total validation images: {self.valid_generator.samples}")
        print(f"📂 Classes found: {len(self.train_generator.class_indices)}")
        print(f"📂 Class indices: {self.train_generator.class_indices}")

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        """
        Save model in modern `.keras` format.
        """
        save_path = str(path).replace(".h5", ".keras")  # ✅ ensure keras format
        model.save(save_path)
        print(f"✅ Model saved at: {save_path}")

    def train(self, callback_list: list):
        """
        Train the model with generators, handle class imbalance,
        and save the best version in `.keras` format.
        """
        # ---- Compute steps ----
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        # ---- Compute class weights for imbalance ----
        df = pd.read_csv(self.config.label_csv_path)

        le = LabelEncoder()
        df['label_encoded'] = le.fit_transform(df['label'])

        class_weights = compute_class_weight(
            class_weight="balanced",
            classes=np.unique(df['label_encoded']),
            y=df['label_encoded']
        )
        class_weight_dict = {i: w for i, w in enumerate(class_weights)}
        print("\n📊 Computed class weights:", class_weight_dict)

        # ---- Add EarlyStopping + ReduceLROnPlateau + ModelCheckpoint + TensorBoard ----
        early_stopping = EarlyStopping(
            monitor="val_loss",
            patience=6,
            restore_best_weights=True
        )

        reduce_lr = ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.2,
            patience=3,
            min_lr=1e-7
        )

        model_checkpoint = ModelCheckpoint(
            filepath=str(self.config.trained_model_path).replace(".h5", "_best.keras"),
            monitor="val_loss",
            save_best_only=True,
            verbose=1
        )

        # 🔹 TensorBoard callback
        log_dir = os.path.join("logs", time.strftime("run_%Y%m%d-%H%M%S"))
        tensorboard_cb = TensorBoard(
            log_dir=log_dir,
            histogram_freq=1,
            write_graph=True,
            write_images=True
        )
        print(f"📊 TensorBoard logs saved at: {log_dir}")

        # Merge user-provided callbacks with these
        callback_list = callback_list + [early_stopping, reduce_lr, model_checkpoint, tensorboard_cb]

        # ---- Train the model ----
        history = self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator,
            callbacks=callback_list,
            class_weight=class_weight_dict
        )

        # ---- Save final trained model ----
        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )
        return history


In [10]:
try:
    # Load configs
    config = ConfigurationManager()
    training_config = config.get_training_config()
    prepare_callbacks_config = config.get_prepare_callbacks_config()

    # Prepare callbacks (e.g., TensorBoard, EarlyStopping)
    prepare_callbacks = PrepareCallback(config=prepare_callbacks_config)
    callback_list = prepare_callbacks.get_tb_ckpt_callbacks()

    # Initialize Training
    training = Training(config=training_config)

    # Load and recompile base model
    training.get_base_model()

    # Prepare generators
    training.train_valid_generator()

    # Train and save best model (.keras format)
    training.train(callback_list=callback_list)

except Exception as e:
    raise e


[2025-09-30 10:59:57,510: INFO: common: Directory created at: artifacts]
[2025-09-30 10:59:57,517: INFO: common: Directory created at: artifacts\training]
[2025-09-30 10:59:57,522: INFO: common: Directory created at: artifacts\prepare_callbacks]
[2025-09-30 10:59:57,526: INFO: common: Directory created at: artifacts\prepare_callbacks\checkpoints]
[2025-09-30 10:59:57,532: INFO: common: Directory created at: artifacts\prepare_callbacks\tensorboard_logs]


[2025-09-30 10:59:58,403: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
✅ Model compiled with BatchNorm + Dropout + L2 regularization.

📊 Class Distribution in Dataset:
label
Salmonella            2625
Coccidiosis           2476
Healthy               2404
New Castle Disease     562
Name: count, dtype: int64
Found 1613 validated image filenames belonging to 4 classes.
Found 6454 validated image filenames belonging to 4 classes.

🖼️ Total training images: 6454
🖼️ Total validation images: 1613
📂 Classes found: 4
📂 Class indices: {'Coccidiosis': 0, 'Healthy': 1, 'New Castle Disease': 2, 'Salmonella': 3}

📊 Computed class weights: {0: np.float64(0.8145193861066236), 1: np.float64(0.838914309484193), 2: np.float64(3.588523131672598), 3: np.float64(0.7682857142857142)}
📊 TensorBoard logs saved at: logs\run_20250930-105959


c:\Users\MATT\Documents\End-to-end-chicken-Disease-Classification\venv\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
c:\Users\MATT\Documents\End-to-end-chicken-Disease-Classification\venv\Lib\site-packages\tensorflow\python\data\ops\structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


Epoch 1/20
403/403 ━━━━━━━━━━━━━━━━━━━━ 0s 203s/step - accuracy: 0.6177 - loss: 1.1495  
Epoch 1: val_loss improved from None to 0.71296, saving model to artifacts\prepare_callbacks\checkpoints\best_model.keras

Epoch 1: val_loss improved from None to 0.71296, saving model to artifacts\training\trained_model_best.keras
403/403 ━━━━━━━━━━━━━━━━━━━━ 81726s 203s/step - accuracy: 0.6839 - loss: 0.9903 - val_accuracy: 0.7669 - val_loss: 0.7130 - learning_rate: 0.0010
Epoch 2/20
  1/403 ━━━━━━━━━━━━━━━━━━━━ 1:15:04 11s/step - accuracy: 0.5625 - loss: 1.1465

c:\Users\MATT\Documents\End-to-end-chicken-Disease-Classification\venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 2: val_loss improved from 0.71296 to 0.62059, saving model to artifacts\prepare_callbacks\checkpoints\best_model.keras

Epoch 2: val_loss improved from 0.71296 to 0.62059, saving model to artifacts\training\trained_model_best.keras
403/403 ━━━━━━━━━━━━━━━━━━━━ 282s 675ms/step - accuracy: 0.5625 - loss: 1.1465 - val_accuracy: 0.7956 - val_loss: 0.6206 - learning_rate: 0.0010
Epoch 3/20
403/403 ━━━━━━━━━━━━━━━━━━━━ 0s 11s/step - accuracy: 0.7421 - loss: 0.8607 
Epoch 3: val_loss improved from 0.62059 to 0.55996, saving model to artifacts\prepare_callbacks\checkpoints\best_model.keras

Epoch 3: val_loss improved from 0.62059 to 0.55996, saving model to artifacts\training\trained_model_best.keras
403/403 ━━━━━━━━━━━━━━━━━━━━ 4873s 12s/step - accuracy: 0.7457 - loss: 0.8535 - val_accuracy: 0.8319 - val_loss: 0.5600 - learning_rate: 0.0010
Epoch 4/20
  1/403 ━━━━━━━━━━━━━━━━━━━━ 1:12:03 11s/step - accuracy: 0.7500 - loss: 1.2268
Epoch 4: val_loss did not improve from 0.55996

Epoch 4: